# Checkpoint-selection sweep (A100, ~30-40 min)

Evaluates historical training checkpoints (git revisions of the rolling
`last-checkpoint` folder in `isaacmg/qwen3-vl-8b-hebrew-ckpt`) on 30 held-out
val crops each, producing a CER-vs-step selection curve. Pick the winner for
the final merge instead of assuming the last checkpoint is best.

Colab secrets needed: `HF_TOKEN`.

In [ ]:
# Cell 1 — installs + auth
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
%pip install -q bitsandbytes peft rapidfuzz hf_transfer python-Levenshtein
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
import torch
assert torch.cuda.is_available()

In [ ]:
# Cell 2 — pick checkpoint revisions (commit per "step N, checkpoint" push)
import re
from huggingface_hub import list_repo_commits

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-ckpt"
TARGET_STEPS = [500, 1000, 1500, 2000, 2400, 2900, 3400, 3900, 4400, 4686]

commits = list_repo_commits(CKPT_REPO)
by_step = {}
for c in commits:
    m = re.search(r"step (\d+), checkpoint", c.title)
    if m:
        by_step.setdefault(int(m.group(1)), c.commit_id)
steps = [s for s in TARGET_STEPS if s in by_step] or sorted(by_step)[-10:]
print(f"{len(by_step)} checkpoint commits; sweeping steps: {steps}")

In [ ]:
# Cell 3 — eval samples: same seeded selection as the local quick_eval
import random
from datasets import load_dataset

SEED, N_SAMPLES = 20260714, 30
val = load_dataset("isaacmg/talmud_finetune", split="val")
val = val.filter(lambda t: t == "crop_transcribe", input_columns="task")
idx = list(range(len(val)))
random.Random(SEED).shuffle(idx)
samples = [val[i] for i in idx[:N_SAMPLES]]
print(f"{len(samples)} eval samples,",
      {s: sum(1 for x in samples if x["section"] == s)
       for s in ("gemara", "rashi", "tosafot")})

In [ ]:
# Cell 4 — base model once (plain HF 4-bit; no training, inference only)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MIN_PIX, MAX_PIX = 256 * 28 * 28, 4_500_000
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_quant_type="nf4")
base = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", quantization_config=bnb, device_map="cuda:0",
)
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)

In [ ]:
# Cell 5 — sweep: attach each checkpoint's adapter -> CER on the 30 crops
import re as _re
import statistics
from Levenshtein import distance
from huggingface_hub import snapshot_download
from peft import PeftModel

def norm(t):
    return _re.sub(r"\s+", " ", t).strip()

def cer(hyp, ref):
    return distance(norm(hyp), norm(ref)) / max(len(norm(ref)), 1)

@torch.inference_mode()
def transcribe(model, sample):
    msgs = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": sample["question"]},
    ]}]
    text = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(images=sample["image"], text=text,
                       return_tensors="pt").to("cuda:0")
    out = model.generate(**inputs, max_new_tokens=2200, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]

results = {}
for step in steps:
    local = snapshot_download(CKPT_REPO, revision=by_step[step],
                              allow_patterns="last-checkpoint/*",
                              local_dir=f"ckpts/{step}")
    model = PeftModel.from_pretrained(base, f"ckpts/{step}/last-checkpoint")
    model.eval()
    per_section = {}
    for s in samples:
        c = cer(transcribe(model, s), s["answer"])
        per_section.setdefault(s["section"], []).append(c)
    summary = {k: round(statistics.mean(v), 3) for k, v in sorted(per_section.items())}
    summary["mean"] = round(statistics.mean(x for v in per_section.values() for x in v), 3)
    # reads = samples with CER < 0.5 (genuine transcription attempts)
    summary["reads"] = sum(1 for v in per_section.values() for x in v if x < 0.5)
    results[step] = summary
    print(f"step {step:5d}: {summary}")
    base_restored = model.unload()   # strip adapter, restore clean base
    del model

print()
print("step | gemara | rashi | tosafot | mean | reads/30")
for step, r in results.items():
    print(f"{step:5d} | {r.get('gemara','-'):6} | {r.get('rashi','-'):5} |"
          f" {r.get('tosafot','-'):7} | {r['mean']:5} | {r['reads']}")
best = min(results, key=lambda s: results[s]["mean"])
print(f"\nBest mean CER: step {best} -> use this revision for the final merge:")
print(f"  {by_step[best]}")